# GWAS RSS fine-mapping (per study × per LD block)

## Description

End-to-end SuSiE-RSS fine-mapping over GWAS summary statistics. Four SoS steps:

1. **`[generate_manifest]`** &mdash; calls `gwas_rss_manifest.R` to resolve `--gwas-meta` + `--gwas-tsv-list` against `--region-list` + `--regions` into a single TSV manifest (one row per study x region job). All cross-product / dedup / sanitisation logic lives in R.
2. **`[generate_gwas_sumstats]`** &mdash; per (study × LD block) fan-out. Calls `gwas_sumstats_construct.R` to build one `GwasSumStats` RDS per study × block, with `summaryStatsQc()` (optionally including SLALOM/DENTIST QC and RAISS imputation).
3. **`[gwas_fine_mapping]`** &mdash; per RDS, runs `fine_mapping.R --gwas-sumstats` which dispatches to `pecotmr::fineMappingPipeline(gwasSumStats, methods = ...)` (default `susie`; the SuSiE-RSS variant inside pecotmr).
4. **`[gwas_rss_plot]`** (optional, opt-out via `--no-plot`) &mdash; per fine-mapping RDS, renders a PIP plot via `gwas_rss_plot.R`.

Replaces the legacy `mnm_analysis/mnm_methods/rss_analysis.ipynb` two-step workflow. Reuses `fine_mapping.R` (no new fine-mapping worker required) and the new `--method-args` / `--qc-args` JSON passthroughs for per-method and per-QC tuning.

## Inputs

### Per-study inputs (one or both forms; the workflow fans out over their union)

- `--gwas-meta` &mdash; TSV with columns `study_id`, `path` (relative to the meta file or absolute), and optionally `column_mapping` (path to a per-study YAML; see `gwas_sumstats_construct.R --column-mapping`). One row per study.
- `--gwas-tsv-list S1=path1 S2=path2 ...` &mdash; explicit STUDY=PATH pairs (alternative to / additional to `--gwas-meta`).

### Region inputs (one or both; their union is the fan-out target)

- `--region-list` &mdash; BED-like TSV (`#chr`, `start`, `end`, ...) listing LD blocks.
- `--regions chr:start-end ...` &mdash; explicit region strings.

### LD reference

- `--ld-meta` &mdash; LD-reference meta TSV (`#chr`, `start`, `end`, `path`). One LD block per row, or one row with `start = end = 0` to mean &ldquo;whole chromosome.&rdquo;

### QC knobs (forwarded to `gwas_sumstats_construct.R`)

- `--qc-method` &mdash; `none` (default), `slalom`, or `dentist`.
- `--impute` &mdash; flag to enable RAISS sumstat imputation.
- `--qc-args '{"key":value, ...}'` &mdash; JSON passthrough for any other `summaryStatsQc()` kwarg.

### Fine-mapping knobs (forwarded to `fine_mapping.R --gwas-sumstats`)

- `--methods` &mdash; comma-separated method tokens. Default `susie` (= SuSiE-RSS internally for `GwasSumStats` input).
- `--coverage` &mdash; SuSiE credible-set coverage. Default 0.95.
- `--method-args '{"susie":{"L":1}}'` &mdash; per-method kwargs spliced into `fineMappingPipeline()`.

### Infrastructure

- `--cwd` &mdash; output directory (default `output`).
- `--modular-script-dir` &mdash; directory holding the per-task R workers (default `code/script`).
- `--genome` &mdash; genome build label (default `GRCh38`).
- `--no-plot` &mdash; skip the PIP plot step.

## Outputs

- `{cwd}/sumstats/{study}.{chrN_S_E}.gwas_sumstats.rds` &mdash; one per (study, LD block).
- `{cwd}/fine_mapping/{study}.{chrN_S_E}.gwas_finemap.rds` &mdash; one `GwasFineMappingResult` per (study, LD block).
- `{cwd}/plots/{study}.{chrN_S_E}.pip_plot.png` &mdash; per (study, LD block) PIP plot (when `--no-plot` is unset).


## Example

Minimal run with explicit inputs:
```bash
sos run pipeline/gwas_rss_fine_mapping.ipynb gwas_rss_fine_mapping \
    --cwd output --modular-script-dir /path/to/xqtl-protocol/code/script \
    --gwas-tsv-list MyGWAS=input/twas/protocol_example.twas.gwas_sumstats.chr22.tsv.gz \
    --regions chr22:10516173-17414263 \
    --ld-meta input/rss_analysis/protocol_example.ld_meta.tsv
```

With SLALOM QC + RAISS imputation + per-method susie tuning:
```bash
sos run pipeline/gwas_rss_fine_mapping.ipynb gwas_rss_fine_mapping \
    --cwd output --modular-script-dir /path/to/code/script \
    --gwas-meta input/gwas/gwas_meta.tsv \
    --region-list input/rss_analysis/ld_blocks.bed \
    --ld-meta input/rss_analysis/ld_meta.tsv \
    --qc-method slalom --impute --qc-args '{"mafCutoff":0.01}' \
    --methods susie --method-args '{"susie":{"L":10}}'
```


In [ ]:
[global]
parameter: cwd = path('output')
parameter: modular_script_dir = path('code/script')
# --- per-study GWAS inputs (use either or both) ---------------------
parameter: gwas_meta = path('.')
parameter: gwas_tsv_list = []   # list of STUDY=PATH items
# --- region inputs (use either or both) -----------------------------
parameter: region_list = path('.')
parameter: regions = []         # list of chr:start-end strings
# --- LD reference + genome ------------------------------------------
parameter: ld_meta = path
parameter: genome = 'GRCh38'
# --- QC knobs (forwarded to gwas_sumstats_construct.R) --------------
parameter: qc_method = 'none'   # none | slalom | dentist
parameter: impute = False
parameter: qc_args = ''         # JSON object spliced into summaryStatsQc()
# --- fine-mapping knobs (forwarded to fine_mapping.R) ---------------
parameter: methods = 'susie'
parameter: coverage = 0.95
parameter: secondary_coverage = '0.7,0.5'
parameter: min_abs_corr = 0.8
parameter: median_abs_corr = ''   # empty -> off (OR-logic purity; needs pecotmr step-1)
parameter: pip_cutoff = 0.025
parameter: L = 20
parameter: L_greedy = 5
parameter: method_args = ''     # nested per-method JSON object
# --- plot opt-out ---------------------------------------------------
parameter: no_plot = False
# --- output prefix for the manifest file ----------------------------
parameter: manifest_name = 'gwas_rss'
# --- infrastructure -------------------------------------------------
parameter: container = ''
parameter: job_size = 1
parameter: walltime = '2h'
parameter: mem = '16G'
parameter: numThreads = 1

In [ ]:
[generate_manifest]
# Resolve --gwas-meta / --gwas-tsv-list x --region-list / --regions into a
# single manifest TSV via gwas_rss_manifest.R. Downstream steps fan out
# over its rows; no Python parsing in this notebook.
input: None
output: f"{cwd}/{manifest_name}.manifest.tsv"
task: trunk_workers = 1, trunk_size = 1, walltime = '15m', mem = '2G', cores = 1, tags = f"{step_name}_{_output:bn}"
bash: expand = '${ }', stderr = f"{_output}.stderr", stdout = f"{_output}.stdout", container = container
    Rscript ${modular_script_dir}/pecotmr_integration/gwas_rss_manifest.R \
        ${('--gwas-meta ' + str(gwas_meta)) if gwas_meta.is_file() else ''} \
        ${('--gwas-tsv-list ' + ' '.join(str(x) for x in gwas_tsv_list)) if gwas_tsv_list else ''} \
        ${('--region-list ' + str(region_list)) if region_list.is_file() else ''} \
        ${('--regions ' + ' '.join(str(x) for x in regions)) if regions else ''} \
        --output ${_output}

In [ ]:
[generate_gwas_sumstats]
# Fan out over the manifest's rows: one (study, region) GwasSumStats per
# row. Manifest columns: study_id, gwas_tsv, column_mapping, chr, start,
# end, region_id, gwas_tsv_basename.
import csv
jobs = list(csv.DictReader(open(f"{cwd}/{manifest_name}.manifest.tsv"), delimiter='\t'))
input: for_each = 'jobs'
output: f"{cwd}/sumstats/{_jobs['study_id']}.{_jobs['region_id']}.gwas_sumstats.rds"
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f"{step_name}_{_output:bn}"
bash: expand = '${ }', stderr = f"{_output}.stderr", stdout = f"{_output}.stdout", container = container
    Rscript ${modular_script_dir}/pecotmr_integration/gwas_sumstats_construct.R \
        --study ${_jobs['study_id']} \
        --gwas-tsv ${_jobs['gwas_tsv']} \
        --ld-block ${_jobs['chr']}:${_jobs['start']}-${_jobs['end']} \
        --ld-meta ${ld_meta} \
        --genome ${genome} \
        ${('--column-mapping ' + _jobs['column_mapping']) if _jobs['column_mapping'] else ''} \
        --qc-method ${qc_method} \
        ${'--impute' if impute else ''} \
        ${('--qc-args ' + repr(qc_args)) if qc_args else ''} \
        --output ${_output}

In [ ]:
[gwas_fine_mapping]
# Per-RDS fan-out: one fine-mapping task per (study, region) GwasSumStats.
output: f"{cwd}/fine_mapping/{_input:bnn}.gwas_finemap.rds", group_by = 1
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f"{step_name}_{_output:bn}"
bash: expand = '${ }', stderr = f"{_output}.stderr", stdout = f"{_output}.stdout", container = container
    Rscript ${modular_script_dir}/pecotmr_integration/fine_mapping.R \
        --gwas-sumstats ${_input} \
        --methods ${methods} \
        --coverage ${coverage} \
        --secondary-coverage ${secondary_coverage} \
        --min-abs-corr ${min_abs_corr} \
        --pip-cutoff ${pip_cutoff} \
        --L ${L} \
        --L-greedy ${L_greedy} \
        ${('--median-abs-corr ' + str(median_abs_corr)) if str(median_abs_corr) != '' else ''} \
        ${('--method-args ' + repr(method_args)) if method_args else ''} \
        --output ${_output}

In [ ]:
[gwas_rss_plot]
stop_if(no_plot, '--no-plot set; skipping PIP plot step.')
output: f"{cwd}/plots/{_input:bnn}.pip_plot.png", group_by = 1
task: trunk_workers = 1, trunk_size = job_size, walltime = '30m', mem = '4G', cores = 1, tags = f"{step_name}_{_output:bn}"
bash: expand = '${ }', stderr = f"{_output}.stderr", stdout = f"{_output}.stdout", container = container
    Rscript ${modular_script_dir}/pecotmr_integration/gwas_rss_plot.R \
        --input ${_input} \
        --output ${_output}